In [137]:
import time

running_time = time.time()

In [138]:
import sys
sys.path.append('../')

import logging
logging.getLogger('matplotlib').setLevel(logging.CRITICAL)
logging.getLogger('graphein').setLevel(logging.INFO)

In [139]:
import pandas as pd
import numpy as np

from tqdm import tqdm
import networkx as nx

import warnings
warnings.filterwarnings("ignore")


In [140]:
from pkg.PDBGraphStore import PDBGraphStore as store
from pkg.MemoryMeasuring import MemoryMeasuring as m

In [141]:
from graphein.protein.config import ProteinGraphConfig
from graphein.protein.edges.distance import (
    add_hydrogen_bond_interactions, 
    add_peptide_bonds, 
    add_k_nn_edges, 
    add_aromatic_interactions,
    add_aromatic_sulphur_interactions
)
from graphein.protein.graphs import construct_graph
from graphein.ml.conversion import GraphFormatConvertor

from graphein.protein.utils import download_pdb
import os

from functools import partial

In [142]:
def fit(graph: nx.Graph) -> nx.Graph:
    g_config = graph.graph["config"]
    g_pdb_code = graph.graph["pdb_code"]
    graph.graph.clear()
    graph.graph['config'] = g_config
    graph.graph['pdb_code'] = g_pdb_code

    return graph

In [143]:
constructors = {
    # "edge_construction_functions": [partial(add_k_nn_edges, k=3, long_interaction_threshold=0)],
    "graph_metadata_functions": [fit],
    "edge_construction_functions": [add_hydrogen_bond_interactions, add_aromatic_sulphur_interactions, add_aromatic_interactions, add_peptide_bonds],
    'granularity': 'CA',
    #"node_metadata_functions": [add_dssp_feature]
}

config = ProteinGraphConfig(**constructors)
print(config.dict())

{'granularity': 'CA', 'keep_hets': [], 'insertions': True, 'alt_locs': 'max_occupancy', 'pdb_dir': None, 'verbose': False, 'exclude_waters': True, 'deprotonate': False, 'protein_df_processing_functions': None, 'edge_construction_functions': [<function add_hydrogen_bond_interactions at 0x7fa8a8598040>, <function add_aromatic_sulphur_interactions at 0x7fa8a8598220>, <function add_aromatic_interactions at 0x7fa8a8598180>, <function add_peptide_bonds at 0x7fa8a857be20>], 'node_metadata_functions': [<function meiler_embedding at 0x7fa8a859ad40>], 'edge_metadata_functions': None, 'graph_metadata_functions': [<function fit at 0x7fa6bf15b6a0>], 'get_contacts_config': None, 'dssp_config': None}


In [144]:
df = pd.read_csv(
    '../../data/scope.2.08.txt',
    sep="\t",
    comment="#",
    header=None,
)

df.columns = [
    "domain_id",
    "pdb",
    "region",
    "sccs",
    "sunid",
    "hierarchy"
]

print(len(df))

df.head(3)
df = df.sample(n=3000, random_state=42)

df["superfamily"] = df["sccs"].str.extract(r"^([a-z]\.\d+\.\d+)")

print(df.head(3))
print(len(df))

344851
       domain_id   pdb   region      sccs   sunid  \
168673   d1gytd1  1gyt  D:1-178  c.50.1.1   70771   
328794   d3quca2  3quc    A:0-0   l.1.1.1  294841   
150848   d6b7fa1  6b7f  A:2-159  c.26.1.3  349442   

                                                hierarchy superfamily  
168673  cl=51349,cf=52948,sf=52949,fa=52950,dm=52951,s...      c.50.1  
328794  cl=310555,cf=310573,sf=310607,fa=310682,dm=310...       l.1.1  
150848  cl=51349,cf=52373,sf=52374,fa=52397,dm=52398,s...      c.26.1  
3000


In [145]:
pdb_store = store()

In [146]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

In [147]:
df["superfamily_id"] = encoder.fit_transform(df["superfamily"])

In [148]:
print(df['superfamily_id'].head())
print(len(df))

168673    319
328794    655
150848    300
24057      87
240283    547
Name: superfamily_id, dtype: int64
3000


In [149]:
pdb_data_path = "../../data/pdb_files/"

for idx, pdb_code in enumerate(tqdm(df['pdb'])):
    if os.path.exists(f'{pdb_data_path}/{pdb_code}.pdb'):
        pdb_file = os.path.abspath(f'{pdb_data_path}/{pdb_code}.pdb')
    else:
        try:
            pdb_file = download_pdb(pdb_code, f'{pdb_data_path}/')
        except Exception as e:
            print(str(idx) + "processing error....")
            pass

    if pdb_file == None:
        print(str(idx) + "processing error....")
        pass
    
    try:
        graph = construct_graph(config=config, path=pdb_file, pdb_code=pdb_code)
        pdb_store.insert({pdb_code: graph})

    except Exception as e:
        print(str(idx) + ' processing error...')
        print(e)
        pass

graph = None

  0%|          | 0/3000 [00:00<?, ?it/s]

Output()

  0%|          | 1/3000 [00:01<1:14:32,  1.49s/it]

Output()

Output()

...

100%|█████████▉| 2998/3000 [18:05<00:00,  6.80it/s]

Output()

100%|█████████▉| 2999/3000 [18:05<00:00,  6.07it/s]

Output()

100%|██████████| 3000/3000 [18:05<00:00,  2.76it/s]


In [150]:
# from pkg.MemoryMeasuring import MemoryMeasuring as m

# memo = m(pdb_store)

In [151]:
# print(memo.total_memory())

In [152]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import (
    SAGEConv,
    BatchNorm,
    global_mean_pool
)


class ProteinGraphClassifier(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        num_classes,
        dropout=0.3
    ):
        super().__init__()

        self.conv1 = SAGEConv(input_dim, hidden_dim)
        self.bn1 = BatchNorm(hidden_dim)

        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.bn2 = BatchNorm(hidden_dim)

        self.conv3 = SAGEConv(hidden_dim, hidden_dim)
        self.bn3 = BatchNorm(hidden_dim)

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, data):

        x = torch.cat(
            [
                data.meiler.float(),
                data.coords.float()
            ],
            dim=1
        )
        edge_index = data.edge_index
        batch = data.batch

        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)

        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)

        x = self.conv3(x, edge_index)
        x = self.bn3(x)
        x = F.relu(x)

        x = global_mean_pool(x, batch)

        out = self.classifier(x)

        return out

In [153]:
from sklearn.model_selection import train_test_split

In [154]:
counts = df["superfamily_id"].value_counts()
print(len(df))
valid_families = counts[counts >= 6].index

df = df[df["superfamily_id"].isin(valid_families)]

print(len(df))

3000
2033


In [155]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["superfamily_id"],
    random_state=42
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["superfamily_id"],
    random_state=42
)

temp_df = None

In [156]:
print(len(train_df))
print(len(valid_df))
print(len(test_df))

1423
305
305


In [157]:
print(train_df.head(1))

       domain_id   pdb region     sccs  sunid  \
306554   d1jzym_  1jzy     M:  i.1.1.2  67891   

                                                hierarchy superfamily  \
306554  cl=58117,cf=58118,sf=58119,fa=58124,dm=58125,s...       i.1.1   

        superfamily_id  
306554             638  


In [158]:
convertor = GraphFormatConvertor(src_format="nx", dst_format="pyg", columns=["coords", "edge_index", "meiler"])

In [159]:
import traceback

In [160]:
train_ds = []

for _, row in tqdm(train_df.iterrows(), total=len(train_df)):
    pdb_code = row["pdb"]
    sf_id = row["superfamily_id"]
    try:
        g = pdb_store.extract(pdb_code)
        aux = {}

        aux[pdb_code] = convertor(g)
        aux[pdb_code].y = torch.tensor([sf_id], dtype=torch.long)
        train_ds.append(aux[pdb_code])

    except Exception:
        print(f'Erro em {pdb_code}')
        traceback.print_exc()

100%|██████████| 1423/1423 [05:29<00:00,  4.31it/s]


In [161]:
valid_ds = []

for _, row in tqdm(valid_df.iterrows(), total=len(valid_df)):
    pdb_code = row["pdb"]
    sf_id = row["superfamily_id"]
    try:
        g = pdb_store.extract(pdb_code)
        aux = {}

        aux[pdb_code] = convertor(g)
        aux[pdb_code].y = torch.tensor([sf_id], dtype=torch.long)
        valid_ds.append(aux[pdb_code])

    except Exception:
        print(f'Erro em {pdb_code}')
        traceback.print_exc()

 13%|█▎        | 39/305 [00:10<00:34,  7.64it/s]Traceback (most recent call last):
  File "/tmp/ipykernel_211/3958880290.py", line 7, in <module>
    g = pdb_store.extract(pdb_code)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jovyan/app/src/PDBGraphStore_notebooks/../pkg/PDBGraphStore.py", line 369, in extract
    return extract()
           ^^^^^^^^^
  File "/home/jovyan/app/src/PDBGraphStore_notebooks/../pkg/PDBGraphStore.py", line 352, in extract
    pdb_id = self.__body_parts["pdb_code_to_id"][pdb_upper]
             ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^
KeyError: '4GXV'
 14%|█▍        | 43/305 [00:10<00:25, 10.33it/s]

Erro em 4gxv


100%|██████████| 305/305 [01:13<00:00,  4.14it/s]


In [162]:
test_ds = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    pdb_code = row["pdb"]
    sf_id = row["superfamily_id"]
    try:
        g = pdb_store.extract(pdb_code)
        aux = {}

        aux[pdb_code] = convertor(g)
        aux[pdb_code].y = torch.tensor([sf_id], dtype=torch.long)
        test_ds.append(aux[pdb_code])

    except Exception:
        print(f'Erro em {pdb_code}')
        traceback.print_exc()

100%|██████████| 305/305 [00:57<00:00,  5.28it/s]


In [163]:
from torch_geometric.loader import DataLoader

# Create dataloaders
train_loader = DataLoader(train_ds, batch_size=16, shuffle=False, drop_last=True)
valid_loader = DataLoader(valid_ds, batch_size=16, shuffle=False, drop_last=True)
test_loader = DataLoader(test_ds, batch_size=16, drop_last=True)

In [164]:
num_node_features = 10
num_classes = len(encoder.classes_)
print(num_classes)

model = ProteinGraphClassifier(
    input_dim=num_node_features,
    hidden_dim=128,
    num_classes=num_classes
)

656


In [165]:
criterion = nn.CrossEntropyLoss()

In [166]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

In [167]:
def train(model, loader, optimizer, criterion, device):

    model.train()

    total_loss = 0

    for batch in loader:

        batch = batch.to(device)

        optimizer.zero_grad()

        out = model(batch)

        loss = criterion(out, batch.y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [168]:
@torch.no_grad()
def evaluate(model, loader, device):

    model.eval()

    correct = 0
    total = 0

    for batch in loader:

        batch = batch.to(device)

        out = model(batch)

        pred = out.argmax(dim=1)

        correct += (pred == batch.y).sum().item()
        total += batch.y.size(0)

    return correct / total

In [169]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

for epoch in range(1, 101):

    loss = train(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    train_acc = evaluate(
        model,
        train_loader,
        device
    )

    valid_acc = evaluate(
        model,
        valid_loader,
        device
    )

    print(
        f"Epoch {epoch:03d} | "
        f"Loss {loss:.4f} | "
        f"Train {train_acc:.4f} | "
        f"Valid {valid_acc:.4f}"
    )

Epoch 001 | Loss 5.0961 | Train 0.1740 | Valid 0.1809
Epoch 002 | Loss 4.1643 | Train 0.1804 | Valid 0.1776
Epoch 003 | Loss 4.0536 | Train 0.1818 | Valid 0.1809
Epoch 004 | Loss 3.9862 | Train 0.1740 | Valid 0.1776
Epoch 005 | Loss 3.8920 | Train 0.1783 | Valid 0.1743
Epoch 006 | Loss 3.8577 | Train 0.1797 | Valid 0.1809
Epoch 007 | Loss 3.8055 | Train 0.1832 | Valid 0.1776
Epoch 008 | Loss 3.7392 | Train 0.1889 | Valid 0.1809
Epoch 009 | Loss 3.7105 | Train 0.1889 | Valid 0.1776
Epoch 010 | Loss 3.6474 | Train 0.1832 | Valid 0.1809
Epoch 011 | Loss 3.5954 | Train 0.1925 | Valid 0.1809
Epoch 012 | Loss 3.5756 | Train 0.1882 | Valid 0.1809
Epoch 013 | Loss 3.5181 | Train 0.1875 | Valid 0.1875
Epoch 014 | Loss 3.4682 | Train 0.1868 | Valid 0.1908
Epoch 015 | Loss 3.4575 | Train 0.1918 | Valid 0.1875
Epoch 016 | Loss 3.3891 | Train 0.1882 | Valid 0.1875
Epoch 017 | Loss 3.3519 | Train 0.1847 | Valid 0.1908
Epoch 018 | Loss 3.3124 | Train 0.1875 | Valid 0.1809
Epoch 019 | Loss 3.2749 | Tr

In [170]:
with open('gnn_time.txt', 'a') as f:
    f.write(f'\n{str(time.time() - running_time)}')